<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/08_Uplift_Modeling_Causal_Inference/01_Discount_Optimization_TLearner_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 8: Causal Inference & Uplift Modeling

## 1. The Real Business Problem: The Margin Trap
In Phase 7, we mathematically identified that our "At-Risk Loyalists" belong entirely to the Broad Middle segment. The standard retail reflex is to send a blanket promotional discount (e.g., "$10 off your next $50 basket") to this entire group to win them back.

However, grocery profit margins are notoriously razor-thin (typically 1% to 3%). When a business sends a promotion, every recipient falls into one of four invisible behavioral quadrants:
* **The Persuadables:** Will only return if they receive the coupon. *(High ROI).*
* **The Sure Things:** Were going to return regardless of the coupon. *(Massive margin cannibalization if we send it).*
* **The Lost Causes:** Have permanently switched stores. *(Wasted marketing spend).*
* **The Sleeping Dogs:** Inactive customers who get annoyed by marketing spam and actively unsubscribe. *(Negative ROI).*

Standard machine learning cannot solve this because it only predicts an *outcome* (Will they buy?). To protect profit margins, we must predict *causality* (Would they have bought *without* the coupon?). We need to isolate **The Persuadables**.

## 2. The Solution: T-Learner Architecture
We do not need to run expensive, live randomized control trials. We can synthesize historical campaign data (customers who received discounts vs. those who did not) using a **Two-Learner (T-Learner)** approach.

Instead of building one monolithic neural network, we train two distinct, lightweight regression models to predict 30-day future spend:
* **Model $C$ (Control):** Trained exclusively on historical customers who received NO promotional campaigns. Learns baseline spending behavior.
* **Model $T$ (Treatment):** Trained exclusively on historical customers who DID receive promotional campaigns. Learns incentivized spending behavior.

## 3. The Causal Subtraction & Actionable Output
To evaluate our current At-Risk customers, we pass their data through *both* models simultaneously to calculate the **Individual Treatment Effect (ITE)**:

$$Uplift(x) = \hat{Y}_T(x) - \hat{Y}_C(x)$$

* If Model T predicts $60 spend and Model C predicts $10 spend, the Uplift is **+$50**. (A Persuadable).
* If Model T predicts $80 spend and Model C predicts $80 spend, the Uplift is **$0**. (A Sure Thing).

**The Final Business Rule:** We eliminate margin cannibalization by establishing a strict financial threshold. We will only issue the $10 discount to customers where their predicted $Uplift$ strictly exceeds $10.

## Step 1: Engineering the Causal Dataset (Timeline Split)

To predict causality, we must prevent data leakage by strictly separating our observation period from our target measurement period.

* **The Cutoff:** We separate the 1-year dataset by reserving the final 90 days as the Target Window.
* **The Features (X):** Using the pre-cutoff data, we calculate each household's historical spend and trip frequency.
* **The Treatment (T):** We query the `campaigns` table. If a household received marketing prior to the cutoff, they are assigned to the Treatment group (`T = 1`). Otherwise, they belong to the Control group (`T = 0`).
* **The Target (Y):** We calculate the total revenue generated by each household strictly within the final 90-day Target Window.

In [1]:
!pip install completejourney_py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 28.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
from completejourney_py import get_data

print("Fetching raw tables...")
data = get_data()
transactions = data['transactions']
campaigns = data['campaigns']

# 1. Enforce date formatting and define the strict chronological cutoff
transactions['transaction_timestamp'] = pd.to_datetime(transactions['transaction_timestamp'])
max_date = transactions['transaction_timestamp'].max()
cutoff_date = max_date - pd.Timedelta(days=90)

print(f"Dataset split! Observation ends on {cutoff_date.date()}. Target Window is the final 90 days.")

# 2. Extract The Observation Window (Pre-Cutoff)
pre_cutoff_data = transactions[transactions['transaction_timestamp'] < cutoff_date]

# Build baseline customer features (X)
causal_df = pre_cutoff_data.groupby('household_id').agg(
    Historical_Spend=('sales_value', 'sum'),
    Historical_Trips=('basket_id', 'nunique')
).reset_index()

# 3. Define the Treatment Vector (T)
# Identify households that received a campaign during the observation window
treated_households = campaigns['household_id'].unique()
causal_df['Treatment_Flag'] = causal_df['household_id'].isin(treated_households).astype(int)

# 4. Extract The Target Window (Post-Cutoff)
post_cutoff_data = transactions[transactions['transaction_timestamp'] >= cutoff_date]

# Calculate the exact revenue generated in the target window (Y)
future_revenue = post_cutoff_data.groupby('household_id').agg(
    Target_90_Day_Spend=('sales_value', 'sum')
).reset_index()

# 5. Merge into the final analytical dataframe
causal_df = causal_df.merge(future_revenue, on='household_id', how='left')

# If a customer didn't shop in the final 90 days, their future spend is $0
causal_df['Target_90_Day_Spend'] = causal_df['Target_90_Day_Spend'].fillna(0)

print("\n✅ Causal Dataset Engineered:")
display(causal_df.head(10))
display(causal_df['Treatment_Flag'].value_counts().rename(index={0: "Control (0)", 1: "Treatment (1)"}))

Fetching raw tables...
Dataset split! Observation ends on 2017-10-03. Target Window is the final 90 days.

✅ Causal Dataset Engineered:


,household_id,Historical_Spend,Historical_Trips,Treatment_Flag,Target_90_Day_Spend
0,1,1796.67,38,1,618.89
1,2,693.43,13,1,330.69
2,3,951.13,17,1,75.50
3,4,300.21,14,1,141.93
4,5,236.69,16,0,62.98
5,6,2764.07,119,1,700.80
6,7,1506.70,21,1,445.67
7,8,2325.32,49,1,755.49
8,9,418.51,7,0,202.22
9,10,29.96,1,0,0.00


,count
Treatment_Flag,
Treatment (1),1558
Control (0),889


We have 1,558 households in the Treatment group and 889 in the Control group. This is a highly robust statistical split. We have a large enough baseline (889) for the algorithm to perfectly understand normal spending behavior, and an even larger group (1,558) to map exactly how incentives alter that behavior.

Now we build the T-Learner. We will split this dataframe in half, train a distinct Random Forest algorithm on each half, and then pit the models against each other to calculate the exact dollar amount of the Individual Treatment Effect (Uplift).

## Step 2: Training the T-Learner (Causal Subtraction)

We deploy a Two-Learner architecture using Random Forest regressors.

1. **Model C** is trained exclusively on the 889 Control households to map baseline, organic spend.
2. **Model T** is trained exclusively on the 1,558 Treatment households to map incentivized spend.

Once trained, we pass the entire customer base through both models. By subtracting the Model C prediction from the Model T prediction, we calculate the exact **Uplift Score** (the isolated financial value of the coupon).

In [3]:
from sklearn.ensemble import RandomForestRegressor

# 1. Define our X (features) and Y (target)
X_cols = ['Historical_Spend', 'Historical_Trips']
y_col = 'Target_90_Day_Spend'

# 2. Isolate the two realities
df_control = causal_df[causal_df['Treatment_Flag'] == 0]
df_treatment = causal_df[causal_df['Treatment_Flag'] == 1]

# 3. Initialize the Two Learners
model_c = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model_t = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# 4. Train Model C (Baseline behavior)
print("⚙️ Training Model C (Control)...")
model_c.fit(df_control[X_cols], df_control[y_col])

# 5. Train Model T (Incentivized behavior)
print("⚙️ Training Model T (Treatment)...")
model_t.fit(df_treatment[X_cols], df_treatment[y_col])

# 6. Calculate Individual Treatment Effect (Uplift) for ALL customers
print("🧮 Calculating Individual Treatment Effects (Uplift)...")
causal_df['Predicted_Spend_Control'] = model_c.predict(causal_df[X_cols])
causal_df['Predicted_Spend_Treatment'] = model_t.predict(causal_df[X_cols])

# The Causal Subtraction: Uplift = Treatment - Control
causal_df['Uplift_Score'] = causal_df['Predicted_Spend_Treatment'] - causal_df['Predicted_Spend_Control']

# 7. Identify the "Persuadables"
# We sort by Uplift Score descending to find the customers who react best to marketing
persuadables = causal_df.sort_values(by='Uplift_Score', ascending=False)

print("\n🎯 Top 10 Most Persuadable Customers:")
display(persuadables[['household_id', 'Predicted_Spend_Control', 'Predicted_Spend_Treatment', 'Uplift_Score']].head(10))

⚙️ Training Model C (Control)...
⚙️ Training Model T (Treatment)...
🧮 Calculating Individual Treatment Effects (Uplift)...

🎯 Top 10 Most Persuadable Customers:


,household_id,Predicted_Spend_Control,Predicted_Spend_Treatment,Uplift_Score
1937,1975,1272.555750,2484.756223,1212.200473
1407,1430,1272.555750,2442.739617,1170.183867
2273,2322,1655.938760,2762.463532,1106.524772
392,400,1533.249150,2548.185909,1014.936759
695,707,1379.609375,2361.873151,982.263776
959,973,1445.197547,2405.735240,960.537693
2299,2351,1272.555750,2229.951128,957.395378
1209,1229,1272.555750,2227.159847,954.604097
968,982,1272.555750,2223.388594,950.832844
684,696,1272.555750,2214.526458,941.970708


***Looking at Household 1975 at the very top of our list:***

Model C predicts that if we do absolutely nothing, this customer will spend $1,272.56 in the next 90 days. But Model T mathematically proves that if we send them a marketing campaign, their spend skyrockets to $2,484.76. That is an isolated Uplift of $1,212.20.

This is the ultimate "Persuadable" customer. A simple marketing touchpoint unlocks over a thousand dollars in incremental revenue that we would have otherwise lost.

To finalize this causal inference phase, we must deliver on the promise we made in the markdown: Eliminating the Margin Trap. We need to bridge Phase 7 (Churn) and Phase 8 (Causal Inference) to generate the final, definitive list of customers who should actually receive the $10 retention coupon.

## Step 3: Deploying the Margin Protection Rule

In Phase 7, we identified our "At-Risk Loyalists" (highly loyal customers in the Broad Middle segment whose $P(Alive)$ has dropped below 50%).

To protect our profit margins, we will not blanket-email this group. We now apply our Causal Inference model to this specific cohort and enforce a strict financial threshold: **We will only issue a $10 retention coupon to an At-Risk Loyalist if their predicted Causal Uplift is strictly greater than $10.**

In [7]:
!pip install lifetimes

from lifetimes.utils import summary_data_from_transaction_data
from lifetimes import BetaGeoFitter
import warnings
warnings.filterwarnings("ignore")

print("🌉 Bridging Phase 7 data into active memory...")

# 1. Regenerate the RFM Matrix
rfm = summary_data_from_transaction_data(
    transactions, 'household_id', 'transaction_timestamp', 'sales_value', freq='D'
)

# 2. Refit the BG/NBD probability model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(rfm['frequency'], rfm['recency'], rfm['T'])
rfm['P_Alive'] = bgf.conditional_probability_alive(rfm['frequency'], rfm['recency'], rfm['T'])

# 3. Load the clusters and merge
clusters = pd.read_csv('master_customers_fully_clustered.csv')
rfm_with_personas = rfm.reset_index().merge(
    clusters[['household_id', 'Hierarchical_Cluster']],
    on='household_id',
    how='inner'
)

# 4. Re-apply the Phase 7 business logic
rfm_with_personas['Persona'] = rfm_with_personas['Hierarchical_Cluster'].replace({
    0: 'Broad Middle', 1: 'Power Shoppers', 2: 'Premium Shoppers'
})
rfm_with_personas['Is_At_Risk_Loyalist'] = (rfm_with_personas['frequency'] >= 5) & (rfm_with_personas['P_Alive'] < 0.5)

print("✅ Bridge complete! `rfm_with_personas` is now in memory.")

🌉 Bridging Phase 7 data into active memory...
✅ Bridge complete! `rfm_with_personas` is now in memory.


Expanding the coupon scope


In [9]:
# 1. Define the cost of our retention campaign coupon
coupon_cost = 10.00

# 2. Merge Causal Uplift data with Persona data (dropping the strict 'Loyalist' requirement)
final_campaign_df = causal_df.merge(
    rfm_with_personas[['household_id', 'Persona', 'P_Alive']],
    on='household_id',
    how='inner'
)

# 3. Apply the Broader Business Logic
# We target anyone in our vulnerable segment (Broad Middle)
# where the coupon generates positive net revenue.
target_audience = final_campaign_df[
    (final_campaign_df['Persona'] == 'Broad Middle') &
    (final_campaign_df['Uplift_Score'] > coupon_cost)
]

# Sort by who generates the highest incremental value
target_audience = target_audience.sort_values(by='Uplift_Score', ascending=False)

print(f"🎯 Expanded Campaign Target List: {len(target_audience)} profitable interventions identified.")

# Let's look at the financial impact of this broader strategy
total_cost = len(target_audience) * coupon_cost
projected_uplift = target_audience['Uplift_Score'].sum()
projected_profit = projected_uplift - total_cost

print(f"💰 Total Campaign Cost: ${total_cost:,.2f}")
print(f"📈 Projected Gross Uplift: ${projected_uplift:,.2f}")
print(f"💵 Projected Net Profit: ${projected_profit:,.2f}\n")

display(target_audience[['household_id', 'P_Alive', 'Uplift_Score']].head(10))

🎯 Expanded Campaign Target List: 204 profitable interventions identified.
💰 Total Campaign Cost: $2,040.00
📈 Projected Gross Uplift: $24,912.84
💵 Projected Net Profit: $22,872.84



,household_id,P_Alive,Uplift_Score
791,2479,0.999901,772.969026
611,1935,0.999891,728.863833
388,1166,0.999912,518.289012
137,392,0.999813,471.613329
380,1147,0.998636,442.629630
747,2328,0.999780,366.732521
542,1710,0.999886,349.407877
395,1179,0.999906,278.522858
237,712,0.999869,245.060742
695,2181,0.999756,233.448448
